### This notebook aligns 1p calcium imaging data with behavioral data and defines behaviors of interest for downstream notebooks to slice neural data corresponding to those behaviors of interest
#### Originally written by WMS for narrow application in 3-chamber social imaging paradigm with cups


Import necessary modules

In [ ]:
import pandas as pd
import numpy as np
import os
import utils
import scipy.io
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import statsmodels.api as sm
import scipy.stats as stats
import pickle
import cv2

Point to directory with prepared data and create new output path to save all analysis results into

In [ ]:
datapath = r"C:\miniscope_files\analyzed\7981\2025_03_25\bento"

res_dir = os.path.join(datapath, "analysis_results")

In [ ]:
if not os.path.isdir(res_dir):
    os.mkdir(res_dir)

Now do the same for the results from this notebook specifically

In [ ]:
#make path if needed
respath = os.path.join(res_dir, "data_prep")
if not os.path.isdir(respath):
    os.mkdir(respath)

Load in C data .csv from session

(note: format should be a .csv with rows = neurons, columns = miniscope frames

In [ ]:
Cdata_file = os.path.join(datapath, "miniscope\Cdata.csv")

Cdata = pd.read_csv(Cdata_file, header=0)

#also define the number of cells in your dataset
numcells = Cdata.shape[0]
print(numcells)

Examine C data

In [ ]:
Cdata

In [ ]:
cell_id=23

In [ ]:
#optional - visualize individual neuron's trace
fig, ax = utils.plot_trace(Cdata,cell_id,downsample=1,data_type='Cdata')

Calculate dF/F -- standardize every neuron's trace to an estimate of the session-wide noise/baseline

In [ ]:
dff, baseline = utils.calculate_dff_median(Cdata)
dff

Visualize dF/F calculation to make sure you agree with how baseline was calculated (non-zero median over all frames of session should work well)

In [ ]:
#define plotting code
def plot_dff_examples(Cdata, dff, baseline, n_neurons=4, seed=None):
    """
    Plot random neurons showing C trace with baseline overlay and resulting dF/F.
    """
    if seed is not None:
        np.random.seed(seed)
    
    neuron_ids = np.random.choice(Cdata.shape[0], size=n_neurons, replace=False)
    
    fig, axes = plt.subplots(n_neurons, 2, figsize=(15, 3*n_neurons), sharex=True)
    
    frames = np.arange(Cdata.shape[1])
    
    for i, neuron in enumerate(neuron_ids):
        # Left column: C trace with baseline
        axes[i, 0].plot(frames, Cdata.iloc[neuron].values, 'k-', linewidth=0.5, alpha=0.8)
        axes[i, 0].axhline(baseline.iloc[neuron, 0], color='r', linewidth=1.5, linestyle='--', label=f'baseline (median)')
        axes[i, 0].set_ylabel(f'Neuron {neuron}\nC')
        if i == 0:
            axes[i, 0].set_title('C trace with baseline')
            axes[i, 0].legend(loc='upper right')
        
        # Right column: dF/F
        axes[i, 1].plot(frames, dff.iloc[neuron].values, 'g-', linewidth=0.5)
        axes[i, 1].axhline(0, color='midnightblue', linewidth=0.5, linestyle='-')
        axes[i, 1].set_ylabel('dF/F')
        if i == 0:
            axes[i, 1].set_title('dF/F')
    
    axes[-1, 0].set_xlabel('Frame')
    axes[-1, 1].set_xlabel('Frame')
    
    plt.tight_layout()
    plt.show()
    
    return neuron_ids  # in case you want to inspect specific neurons

In [ ]:
plotted_neurons = plot_dff_examples(Cdata, dff, baseline, n_neurons=5, seed=None)

Z-scored dF/F

In [ ]:
dfZ = dff.apply(stats.zscore, axis=1, result_type='broadcast')

In [ ]:
dfZ

Compare original trace to z-scored dF/F to non z-scored dF/F sure no wonkiness happened

In [ ]:
cell_id=23

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

frames = np.arange(len(Cdata.columns))

axes[0].plot(frames, dff.iloc[cell_id].values, 'r-', linewidth=1)
axes[0].set_ylabel('original dF/F')

axes[1].plot(frames, dfZ.iloc[cell_id].values, 'b-', linewidth=1)
axes[1].set_ylabel('z-scored dF/F')

fig.suptitle(f'Neuron {cell_id}: original dF/F vs. z-scored dF/F')

plt.tight_layout()
plt.show()

Save out the dF/F and z-scored dF/F as .csv files for downstream use

In [ ]:
dff_file = os.path.join(respath, "dff.csv")
dff.to_csv(dff_file,index=False)

In [ ]:
dfz_file = os.path.join(respath,"dff_zscored.csv")
dfZ.to_csv(dfz_file,index=False)

In [ ]:
Cdata.shape

In [ ]:
dff.shape

In [ ]:
dfZ.shape

#### Also generate just the z-scored C data file if you want to use that instead for some reason

In [ ]:
CdZ = Cdata.apply(stats.zscore, axis=1, result_type='broadcast')
CdZ

In [ ]:
print(Cdata.shape)
print(CdZ.shape)

Plot to make sure z-scored data has similar shape to original data (nothing wonky happened)

In [ ]:
cell_id=15

In [ ]:
fig, ax = utils.plot_trace(Cdata,cell_id,downsample=1,data_type='Cdata')

In [ ]:
fig, ax = utils.plot_trace(CdZ,cell_id,downsample=1,data_type='c_z')

In [ ]:
fig, ax = utils.plot_trace(dfZ,cell_id,downsample=1,data_type='dff_z')

Save out z-scored C data

In [ ]:
cdz_file = os.path.join(respath, "zscored_cdata.csv")
CdZ.to_csv(cdz_file,index=False)

Generate a nice dynamics plot if you want

In [ ]:
respath

In [ ]:
plot_path = os.path.join(respath,"unlabeled_traces.png") #.svg if you want a modifiable plot

In [ ]:
width = 5400 #(3 minutes)
start_frame_plot = 25220
frames = range(start_frame_plot,start_frame_plot+width) #try to leave a ~4000 frame gap
fig, ax = utils.traces_plot(dfZ,n_traces=9,x_range=frames,data_type='dff_z',figsize=(5,3),color=True, #n_traces=9 for random 9 traces (# of colors)
                           offset_scale=9,fps=30,scalebar_time_sec=10,scalebar_ylen=5,save_path=plot_path,show=True)

Produce S data (discritized events)

In [ ]:
events,mags,boolarr = utils.get_transient_timestamps_mod(dfZ, thresh_type="zscore_localMax",std_thresh=3, localMaxNumPoints=100)

In [ ]:
sData = pd.DataFrame(boolarr)
sData = sData.astype(int)
sData

In [ ]:
sData.shape

In [ ]:
cell_id=29
fig, ax = utils.plot_events(dfZ,cell_id,events,mags,downsample=1,trace_type='dff_z')

In [ ]:
#save sData
Sd_file = os.path.join(respath, "sdata.csv")
sData.to_csv(Sd_file,index=False)

Import behavioral data binary rasters

(note: format should be rows = defined behavior labels, columns = behavioral camera frames, and values are binary for whether behavior was occurring during that frame)
(WMS generates these from a Bento-generated .annot file -> annot_to_rast.m script (which calls other functions in matlab version of Bento))

We should probably make this more flexible for different kinds of behavioral inputs but this has been working well for me so far.

In [ ]:
datapath

In [ ]:
raster_file = os.path.join(datapath, "annot_rasters.csv")
raster_df = pd.read_csv(raster_file,index_col=0)

Inspect rasters

In [ ]:
raster_df

In [ ]:
raster_df.shape

Load in the timestamps.csv files from UCLA V4 Miniscope recording software -- the behavioral camera used as well as the miniscope's files

In [ ]:
scope_dir = os.path.join(datapath, "miniscope\\timeStamps.csv")
#cam1_dir = os.path.join(datapath, "minicam1\\timeStamps.csv")
cam2_dir = os.path.join(datapath, "minicam2\\timeStamps.csv")

scope_ts = pd.read_csv(scope_dir)
#cam1_ts = pd.read_csv(cam1_dir)
cam2_ts = pd.read_csv(cam2_dir)

In [ ]:
cam2_ts

Perform sanity check - does raster df have the same # of frames as the timestamps file from the corresponding behavioral camera from this recording session?

In [ ]:
#if cam1_ts.shape[0] == raster_df.shape[1]:
if cam2_ts.shape[0] == raster_df.shape[1]:
    print("Behavioral data loaded in as expected")
else:
    print("WARNING: Behavioral data timestamps frame length does not reflect the # of frames in the annotated behavioral data. Please double-check that timestamps file and behavioral annoptations belong to the same session!")

Sanity check for the neural data as well (belatedly) - does the C data traces have the same # of frames as the miniscope timestamps file?

In [ ]:
if scope_ts.shape[0] == Cdata.shape[1]:
    print("Neural data loaded in as expected")
else:
    print("WARNING: Neural data timestamps frame length does not reflect the # of frames in the C data. Please double-check that timestamps file and neural data pre-processing belong to the same session!")

Define new behaviors defined by combinations of other behaviors

(WMS defines partner and novel interaction as combination of chamber side, trial, and active status)

In [ ]:
datapath

### DOUBLE-CHECK PARTNER SIDE

In [ ]:
#define partner side for first trial!!!
partner_side_social1 = 'right' #'right' or 'left'

In [ ]:
#add partner and novel interaction behaviors
raster_df_updated = utils.add_social_interaction_behaviors(raster_df,partner_side_social1)
raster_df_updated

In [ ]:
#confirming definitions were done correctly
raster_df_updated.loc[:,raster_df_updated.loc['novel_active']==1]

In [ ]:
raster_df_updated.loc[:,raster_df_updated.loc['partner_active']==1]

In [ ]:
respath

In [ ]:
#save to reimport to Bento if you want to visually confirm the new behaviors you've defined
#per Ann - convertToBouts is the bento function that will take a raster and reconvert it to a .annot bout format 
new_raster_file = os.path.join(respath, "annot_rasters_updated.csv")
raster_df_updated.to_csv(new_raster_file,index=True)

If you want sanity check on the annotations outside Bento - load in the behavioral video in the following cell, get a random frame, and compare to the corresponding column of raster_df_updated

In [ ]:
beh_video=os.path.join(datapath, "minicam2\\concat_mergedVideo.avi")

In [ ]:
#run this cell to compare video to annotations lodaded here

#load video
cap = cv2.VideoCapture(beh_video)
    
if not cap.isOpened():
    raise ValueError(f"Could not open video at {beh_video}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if raster_df_updated.shape[1] != total_frames:
    raise ValueError(f"MiniCam video does not have same # of frames as your annotations!")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = np.random.randint(0, min(total_frames, raster_df_updated.shape[1])) #get random frame number

#read specific frame
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ret, frame = cap.read()
cap.release()

if not ret:
    raise ValueError(f"Could not read frame {frame_idx}")

# Convert BGR (OpenCV) to RGB (matplotlib)
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

frame_annots = raster_df_updated.iloc[:, frame_idx]
active_behaviors = frame_annots[frame_annots == 1].index.tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

ax1.imshow(frame_rgb)
ax1.set_title(f'Video Frame {frame_idx}\n({frame_idx/fps:.2f} seconds)', fontsize=12)
ax1.axis('off')

ax2.barh(range(len(raster_df_updated.index)), frame_annots.values, color='steelblue')
ax2.set_yticks(range(len(raster_df_updated.index)))
ax2.set_yticklabels(raster_df_updated.index)
ax2.set_xlabel('Active (1) or Inactive (0)', fontsize=11)
ax2.set_title(f'Behavioral Annotations\nFrame {frame_idx}', fontsize=12)
ax2.set_xlim(-0.1, 1.1)
ax2.axvline(0.5, color='red', linestyle='--', alpha=0.3)

Align and downsample the behavioral recordings to the miniscope recordings (assuming behavioral cameras have faster framerate than miniscope)

In [ ]:
print(f'miniscope timestamps directory: {scope_dir}')
#print(cam1_dir)
print(f'minicam timestamps directory: {cam2_dir}')

In [ ]:
scope_ts

In [ ]:
cam2_ts

In [ ]:
#align files, using miniscope frames as reference
aligned_ts = utils.align_feeds_flexible(scope_dir,cam2=cam2_dir)#cam1=cam1_dir

#inspect
aligned_ts

In [ ]:
if aligned_ts.shape[0] == Cdata.shape[1]:
    print("Feeds aligned and downsampled as expected")
else:
    print("WARNING: Aligned feeds timestamps frame length does not reflect the # of frames in the C data. Please double-check that timestamps file and neural data pre-processing belong to the same session!")

Collect scope frame numbers that should be removed. Specifically, we want to remove any frames that don't have either behavioral data or neural data.

Three possible scenarios:
1) Miniscope frames had an artifact (dust in lens, scanlines, etc.) --> these frames should already have been flagged in pre-processing
2) Miniscope disconnected (either miniscope frame #s are showing up out of order in time or there is a large time gap between frames. The former should have been taken care of in align_feeds_flexible(), cells below will indicate whether the latter is the case. If large disconnections happen, consider using data from this session only before or after the gap)
3) Behavioral camera (minicam) disconnected (either minicam frame #s are showing up out of order in time or there is a large time gap between frames. The former has already been handled by the align_feeds_flexible() function; we will handle the time gap between frames case below)

In [ ]:
#initialize empty list
bad_frames = []

Get approximate largest gap in frames for miniscope. If larger than a few seconds -- consider using data from this session only from before or after this gap

In [ ]:
max_diff_idx = aligned_ts['Time Stamp (ms)'].diff().idxmax()
max_diff_frame = aligned_ts.loc[max_diff_idx, 'Frame Number']
max_diff_value = aligned_ts['Time Stamp (ms)'].diff().max()

print(f"Largest time gap between miniscope frames: {max_diff_value:.1f} ms ({max_diff_value/1000:.2f} sec)")
print(f"Occurs after frame {aligned_ts.loc[max_diff_idx - 1, 'Frame Number']} (before frame {max_diff_frame})")

In [ ]:
datapath

## Pay attention! List any miniscope frames that you flagged as being problematic in pre-processing (due to scan line artifact, etc.) for removal

In [ ]:
bad_frames_preprocessing = range(33817,33820) #[] #add the bad_frames (artifactual from preprocessing) here #[]
bad_frames.extend(bad_frames_preprocessing)
bad_frames

To confirm the bad frames: plot the traces

In [ ]:
frames = range(33700,33850)
fig, ax = utils.traces_plot(dfZ,n_traces=9,x_range=frames,data_type='dff_z',figsize=(5,3),color=False,
                           offset_scale=9,fps=30,scalebar_time_sec=1,scalebar_ylen=5,save_path=False,show=True)

Are any scope frames not monotonously increasing with respect to time (may have been a dropped frame) -- if so (and there's not many), add those frame #s to bad_frames so they don't screw up the normal forward progress of the behavioral data

In [ ]:
if not aligned_ts["Time Stamp (ms)"].is_monotonic_increasing:
    diffs = aligned_ts['Time Stamp (ms)'].diff()
    problematic_rows = aligned_ts[diffs < 0].index-1
    problematic_frame_numbers=aligned_ts.loc[problematic_rows,"Frame Number"].values
    print(f'The following miniscope frames are out of order, consider dropping: {problematic_frame_numbers}')
else:
    print('Scope frames are monotonically increasing')

In [ ]:
#check out those frames in the aligned timestamp structure to decide whether to include:
if not aligned_ts["Time Stamp (ms)"].is_monotonic_increasing:
    print(aligned_ts.loc[problematic_rows])

In [ ]:
#add these frames if need be
if not aligned_ts["Time Stamp (ms)"].is_monotonic_increasing:
    bad_frames.extend(problematic_frame_numbers)

In [ ]:
bad_frames

checking to see whether there are a significant # of duplicated frames in the minicam's column in aligned_ts -- usually if the minicam dropped a long chunk of frames somewhere. We want to drop these C data frames because they don't have real annotations aligned to them so we can't figure out what those traces mean (and we don't want them to bias algorithms if they're mistakenly assigned).

I leave the cutoff for this at 5 minicam frames per miniscope frame, which usally means 10 total minicam frames were dropped -- based on estimated framerates, this should be between a third to a half of a second. This is an acceptable loss of precision of the behavioral data assuming it is rare.

In [ ]:
#First, see how rare the shorter drops are --- if they're very frequent and all over the place, should check the behavioral camera and make a decision for whether to include this session in analysis
#otherwise this session may have had a couple quick lags in the minicam
duplicate_frames = aligned_ts['Frame Number_cam2'].value_counts()
short_matches_a = duplicate_frames[duplicate_frames>1] #at least 2 duplicates
short_matches_b = short_matches_a[short_matches_a<=4] #but no more than 4
short_matches_b.sort_index()

In [ ]:
#now, filter for 5 frames or more
multiple_matches = duplicate_frames[duplicate_frames > 4]
multiple_matches

figure out what is going on with those duplicated frames. Again, it's likely just the minicam dropped some frames -- can confirm by loading video into imageJ or bento and scrubbing across those frames

Let's add the corresponding miniscope frames to the bad_frames list as well.

In [ ]:
if len(multiple_matches) > 0:
    print(f"Found {len(multiple_matches)} minicam frames matched with 5 or more miniscope frames:\n")
    for cam_frame, count in multiple_matches.items():
        mask = aligned_ts['Frame Number_cam2'] == cam_frame
        miniscope_frames = aligned_ts.loc[mask, 'Frame Number'].values
        time_span = aligned_ts.loc[mask, 'Time Stamp (ms)'].max() - aligned_ts.loc[mask, 'Time Stamp (ms)'].min()

        bad_frames.extend(miniscope_frames)
        
        print(f"Minicam frame {cam_frame}: matched to {count} miniscope frames")
        print(f"  Miniscope frames: {miniscope_frames[0]} to {miniscope_frames[-1]}")
        print(f"  Time span: {time_span} ms\n")

Drop all the bad frames from your aligned timestamps df

In [ ]:
aligned_ts_no_drops = aligned_ts.copy()

In [ ]:
#if you decide to drop any 
if len(bad_frames) > 0:
    for frame in bad_frames:
        aligned_ts_no_drops = aligned_ts_no_drops[aligned_ts_no_drops['Frame Number'] != frame]

aligned_ts_no_drops #this is the df to use to filter out only frames with behavioral data for future analysis (from the z-scored  dF/F, etc.)

In [ ]:
unique_bad_frames = np.unique(bad_frames)
if len(unique_bad_frames) == aligned_ts.shape[0] - aligned_ts_no_drops.shape[0]:
    print('Dropped frames successfully')
else:
    print('Frames were not dropped successfully; debug!')

In [ ]:
aligned_ts_file = os.path.join(respath, "aligned_timestamps.csv")
aligned_ts_no_drops.to_csv(aligned_ts_file,index=False)

Get approximate frame rate of miniscope feed (frames/sec) 

In [ ]:
approx_fr = utils.approx_fr(aligned_ts_no_drops)
approx_fr

Slice out your various neural dataframes to only keep the good frames 

Note -- this is using positional indexing (with .iloc) so assumes no columns have been dropped from the original neural data dfs (i.e., they still contain data from all frames of the original miniscope video, which they should at this point if your check for # of timestamps and # of columns above passed)

In [ ]:
scope_frames_to_keep = aligned_ts_no_drops['Frame Number'].to_numpy()

In [ ]:
dff_filtered = dff.iloc[:,scope_frames_to_keep].copy()
dfZ_filtered = dfZ.iloc[:,scope_frames_to_keep].copy()
CdZ_filtered = CdZ.iloc[:,scope_frames_to_keep].copy()
sData_filtered = sData.iloc[:,scope_frames_to_keep].copy()

In [ ]:
dfZ_filtered.shape[1] == scope_frames_to_keep.shape[0]

In [ ]:
dfZ_filtered

In [ ]:
respath

In [ ]:
dff_filt_file = os.path.join(respath, "dff_filtered.csv")
dff_filtered.to_csv(dff_filt_file,index=False)

In [ ]:
dfZ_filt_file = os.path.join(respath, "zscored_dff_filtered.csv")
dfZ_filtered.to_csv(dfZ_filt_file,index=False)

In [ ]:
cdz_filt_file = os.path.join(respath, "zscored_cdata_filtered.csv")
CdZ_filtered.to_csv(cdz_filt_file,index=False)

In [ ]:
sd_filt_file = os.path.join(respath, "sdata_filtered.csv")
sData_filtered.to_csv(sd_filt_file,index=False)

Create a pickle that contains all miniscope frames that were removed, the total # of removed frames, and the percentage of total miniscope frames removed

In [ ]:
drop_dict = {}

drop_dict['number_miniscope_frames_dropped'] = (dfZ.shape[1] - dfZ_filtered.shape[1])
print(drop_dict['number_miniscope_frames_dropped'])
print(len(bad_frames)) # to confirm

drop_dict['fraction_miniscope_frames_removed'] = drop_dict['number_miniscope_frames_dropped']/dfZ.shape[1]
print(drop_dict['fraction_miniscope_frames_removed'])

drop_dict['removed_frames'] = bad_frames

In [ ]:
pickle_path = os.path.join(respath, 'dropped_frames_statistics.pkl')

with open(pickle_path, 'wb') as f:
    pickle.dump(drop_dict,f)

Now, slice out the behavioral data such that only the newly aligned frames are kept and relabel to align with the neural data

In [ ]:
#behavioral annotations

###MAKE SURE TO UPDATE THE CAM NUMBER HERE FOR WHAT WENT INTO EZTRACK
#cam1_frames = aligned_ts_no_drops['Frame Number_cam1'].to_numpy()
cam2_frames = aligned_ts_no_drops['Frame Number_cam2'].to_numpy()
rdf_align = raster_df_updated.iloc[:,cam2_frames].copy()

#relabel the behavioral camera frames to the miniscope frames' labels so the behavioral data can interact with the neural data
scope_frames = aligned_ts_no_drops['Frame Number'].to_numpy()
rdf_align.columns = scope_frames

Inspect the aligned behavioral data (does it have same number of frames as C data, etc.)

In [ ]:
rdf_align

In [ ]:
#looking just at partner frames, for example:
rdf_align.loc[:,rdf_align.loc['partner_active']==1]

Sanity checks of behavioral data - double-checking the annotations that are going to downstream notebooks

In [ ]:
#are parnter and novel active interactions mutually exclusive?

pn_overlap = (rdf_align.loc['partner_active'] == 1) & (rdf_align.loc['novel_active'] == 1)
if pn_overlap.sum() > 0:
    print(f"WARNING: {pn_overlap.sum()} frames have BOTH partner_active and novel_active = 1!")
else:
    print("Confirmed: partner_active and novel_active are mutually exclusive")

In [ ]:
#do active interations with either animal only happen in that animal's chamber?

partner_active_not_in_chamb = ((rdf_align.loc['partner_active'] == 1) & 
                                (rdf_align.loc['partner_chamb'] == 0)).sum()
novel_active_not_in_chamb = ((rdf_align.loc['novel_active'] == 1) & 
                              (rdf_align.loc['novel_chamb'] == 0)).sum()

if partner_active_not_in_chamb > 0 or novel_active_not_in_chamb > 0:
    print(f"WARNING: {partner_active_not_in_chamb} partner_active frames outside partner_chamb")
    print(f"WARNING: {novel_active_not_in_chamb} novel_active frames outside novel_chamb")
else:
    print("Confirmed: active interactions only occur in corresponding chambers")

In [ ]:
#are all trial annotations in order?
empty1_last = rdf_align.columns[rdf_align.loc['empty1'] == 1].max()
social1_first = rdf_align.columns[rdf_align.loc['social1'] == 1].min()
social1_last = rdf_align.columns[rdf_align.loc['social1'] == 1].max()
social2_first = rdf_align.columns[rdf_align.loc['social2'] == 1].min()
social2_last = rdf_align.columns[rdf_align.loc['social2'] == 1].max()
empty2_first = rdf_align.columns[rdf_align.loc['empty2'] == 1].min()

if empty1_last < social1_first < social1_last < social2_first < social2_last < empty2_first:
    print("Confirmed: Trial order is correct (empty1 → social1 → social2 → empty2)")
else:
    print("WARNING: Trial order may be incorrect!")
    print(f"  empty1 ends: {empty1_last}")
    print(f"  social1: {social1_first} to {social1_last}")
    print(f"  social2: {social2_first} to {social2_last}")
    print(f"  empty2 starts: {empty2_first}")

In [ ]:
#Finally, check that partner_active and novel_active only occur during social trials
social_mask = (rdf_align.loc['social1'] == 1) | (rdf_align.loc['social2'] == 1)

partner_outside_social = ((rdf_align.loc['partner_active'] == 1) & ~social_mask).sum()
novel_outside_social = ((rdf_align.loc['novel_active'] == 1) & ~social_mask).sum()

if partner_outside_social == 0 and novel_outside_social == 0:
    print("Confirmed: All partner_active and novel_active frames occur during social trials")
else:
    print(f"WARNING: {partner_outside_social} partner_active frames outside social trials!")
    print(f"WARNING: {novel_outside_social} novel_active frames outside social trials!")

save the aligned behavioral raster

In [ ]:
aligned_raster_file = os.path.join(respath, "aligned_rasters.csv")
rdf_align.to_csv(aligned_raster_file,index=True)

Import XY tracking from ezTrack output ( note --- should modify to be flexible w DLC/SLEAP or Cleversys inputs, as well)

In [ ]:
xy_file = os.path.join(datapath, "ezTrack\\concat_mergedVideo_LocationOutput.csv")

ezt_xy = pd.read_csv(xy_file, header=0)

#convert pixels to millimeters based on median of values provided in ezTrack output
moving_frames = ezt_xy[ezt_xy['Distance_px'] > 0]

if len(moving_frames) == 0:
    raise ValueError("No movement detected in tracking data!")

mmPerPx = (moving_frames['Distance_mm'] / moving_frames['Distance_px']).median()

print(f"Conversion factor: {mmPerPx:.4f} mm/pixel")

ezt_xy['Y'] = ezt_xy['Y']*mmPerPx
ezt_xy['X'] = ezt_xy['X']*mmPerPx

xy = ezt_xy[['X','Y']].T

In [ ]:
xy

In [ ]:
#if cam1_ts.shape[0] == raster_df.shape[1]:
if cam2_ts.shape[0] == xy.shape[1]:
    print("ezTrack data loaded in as expected")
else:
    print("WARNING: Behavioral data timestamps frame length does not reflect the # of frames in the ezTrack data. Please double-check timestamps file and ezTrack data file belong to the same session!")

In [ ]:
#same procedure as above w the behavioral rasters to downsample and align XY position frames with scope frames (as XY position is derived from same feed as behavioral rasters)
##UPDATE CAMERA # HERE
cam2_frames = aligned_ts_no_drops['Frame Number_cam2'].to_numpy()
scope_frames = aligned_ts_no_drops['Frame Number'].to_numpy()

xy_align = xy.iloc[:,cam2_frames].copy()
xy_align.columns = scope_frames

In [ ]:
xy_align

In [ ]:
#save aligned xy file
aligned_xy_file = os.path.join(respath, "aligned_xy.csv")
xy_align.to_csv(aligned_xy_file,index=True)

Defining and saving out social interaction bouts for downstream analysis

1) test animal is actively (instead of passively) interacting w partner or novel

In [ ]:
rdf_align

In [ ]:
#frames where animal is at the partner or novel cup (defined as by the partner or novel cup)
pactframes = rdf_align.loc[:,rdf_align.loc['partner_active']==1]
pactcols = np.asarray(pactframes.columns)
nactframes = rdf_align.loc[:,rdf_align.loc['novel_active']==1]
nactcols = np.asarray(nactframes.columns)

In [ ]:
nactframes

2) test animal is activelyinteracting w either animal at the left or right cup

In [ ]:
#all social frames
social_mask = ((rdf_align.loc['social1']==1) | (rdf_align.loc['social2']==1))
socframes = rdf_align.loc[:,social_mask]
socframes

In [ ]:
#frames during social trials where animal is actively socially interacting in left or right chamber
active_left_frames_social = socframes.loc[:,(socframes.loc['left']==1)&(socframes.loc['active']==1)]
alfcols_social = np.asarray(active_left_frames_social.columns)
active_right_frames_social = socframes.loc[:,(socframes.loc['right']==1)&(socframes.loc['active']==1)]
arfcols_social = np.asarray(active_right_frames_social.columns)

In [ ]:
active_right_frames_social

3. All active interactions in the left or right chamber in the first empty trial (control)

In [ ]:
#all social frames
empty1_mask = (rdf_align.loc['empty1']==1)
e1frames = rdf_align.loc[:,empty1_mask]
e1frames

In [ ]:
#frames during empty trials where animal is actively interacting with left or right cup
active_left_frames_empty1 = e1frames.loc[:,(e1frames.loc['left']==1)&(e1frames.loc['active']==1)]
alfcols_e1 = np.asarray(active_left_frames_empty1.columns)
active_right_frames_empty1 = e1frames.loc[:,(e1frames.loc['right']==1)&(e1frames.loc['active']==1)]
arfcols_e1 = np.asarray(active_right_frames_empty1.columns)

active_left_frames_empty1

In [ ]:
#get the interactions
pact_ints = utils.make_behavior_bouts(pactcols,approx_fr)
nact_ints = utils.make_behavior_bouts(nactcols,approx_fr)
alf_social_ints = utils.make_behavior_bouts(alfcols_social,approx_fr)
arf_social_ints = utils.make_behavior_bouts(arfcols_social,approx_fr)
alf_e1_ints = utils.make_behavior_bouts(alfcols_e1,approx_fr)
arf_e1_ints = utils.make_behavior_bouts(arfcols_e1,approx_fr)

In [ ]:
#save
pact_ints_file = os.path.join(respath, "active_partner_interactions.csv")
pact_ints.to_csv(pact_ints_file,index=True)

In [ ]:
#save
nact_ints_file = os.path.join(respath, "active_novel_interactions.csv")
nact_ints.to_csv(nact_ints_file,index=True)

In [ ]:
#save
alf_social_ints_file = os.path.join(respath, "active_leftcup_interactions_socialtrials.csv")
alf_social_ints.to_csv(alf_social_ints_file,index=True)

In [ ]:
#save
arf_social_ints_file = os.path.join(respath, "active_rightcup_interactions_socialtrials.csv")
arf_social_ints.to_csv(arf_social_ints_file,index=True)

In [ ]:
#save
alf_empty1_ints_file = os.path.join(respath, "active_leftcup_interactions_empty1.csv")
alf_e1_ints.to_csv(alf_empty1_ints_file,index=True)

In [ ]:
#save
arf_empty1_ints_file = os.path.join(respath, "active_rightcup_interactions_empty1.csv")
arf_e1_ints.to_csv(arf_empty1_ints_file,index=True)

Plotting various behaviors

XY position across time

In [ ]:
xyt = xy_align.T
xyt

In [ ]:
#2D lineplot
##NOTE : I think in ezTrack y=0 is at the top and x=0 is at the left

plt.figure()
plt.plot(xyt['X'], xyt['Y'], marker='o', linestyle='-', markersize=2)
plt.title('test animal position, all frames')
plt.xlabel('x')
plt.ylabel('y')
plt.ylim=(300,0)
plt.xlim=(0,1000)
plt.axis('equal')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Assuming your DataFrame is named df and has X and Y positions as rows and time as columns
# Example DataFrame creation (replace this with your actual DataFrame)
# df = pd.DataFrame(data=np.random.rand(2, 91000), index=['X', 'Y'])

# Create a 3D line plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Plot the line in 3D space
ax.plot(xyt['X'], xyt['Y'], range(len(xyt)), marker='o', linestyle='-', markersize=2)

# Set labels for the axes
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.set_zlabel('Time')

# Set Y-axis limits to 0 to 300
ax.set_ylim(0, 300)

ax.set_box_aspect([10, 10, 15])

plt.title('3D Line Plot of XY Coordinates Over Time')
plt.show()

Final sanity checks

In [ ]:
# Verify all filtered outputs have identical shapes and column labels
assert dfZ_filtered.shape[1] == sData_filtered.shape[1] == rdf_align.shape[1] == xy_align.shape[1], \
    "Filtered data shapes don't match!"

print(f"All filtered outputs aligned: {dfZ_filtered.shape[1]} frames")

In [ ]:
print("="*50)
print("SESSION SUMMARY")
print("="*50)
print(f"Animal/Session: {datapath}")
print(f"Partner side (social1): {partner_side_social1}")
print(f"Number of neurons: {numcells}")
print(f"Total miniscope frames: {Cdata.shape[1]}")
print(f"Valid frames after filtering: {dfZ_filtered.shape[1]}")
print(f"Frames dropped: {len(np.unique(bad_frames))} ({drop_dict['fraction_miniscope_frames_removed']*100:.2f}%)")
print(f"Approx frame rate: {approx_fr:.2f} Hz")
print(f"Partner active bouts: {len(pact_ints)}")
print(f"Novel active bouts: {len(nact_ints)}")
print(f"Total partner active time: {pact_ints['length'].sum():.1f} sec")
print(f"Total novel active time: {nact_ints['length'].sum():.1f} sec")
print("="*50)